# 05 — Baja California Fallback Extraction

Extracts text from manually downloaded Baja California PDFs stored in
`data/fallback/Baja_California/` and saves `.txt` files to
`data/raw/Baja_California/`.

Run this notebook when the pipeline's web scraper cannot reach the
ordenjuridico.gob.mx viewer pages for Baja California documents.

## 1. Setup & Imports

In [1]:
import pdfplumber
from pathlib import Path

## 2. Define Source and Output Folders

Place the manually downloaded Baja California PDFs in
`data/fallback/Baja_California/` before running this notebook.
The output directory is created automatically if it does not exist.

In [7]:
BAJA_DIR = Path("../data/bc")
OUT_DIR  = Path("../data/raw/Baja_California")

OUT_DIR.mkdir(parents=True, exist_ok=True)

pdfs = sorted(BAJA_DIR.glob("*.pdf"))
print(f"Source folder : {BAJA_DIR.resolve()}")
print(f"Output folder : {OUT_DIR.resolve()}")
print(f"PDFs found    : {len(pdfs)}")

Source folder : C:\Users\deaqu\OneDrive - Universidad Politécnica de Yucatán\Documentos\Carrera Academica\Data Engineer\7_Quadrimester\Internship\Validition-Pipeline-RAG-Law-and-Compliance-LGBTIQ-Rights\data\bc
Output folder : C:\Users\deaqu\OneDrive - Universidad Politécnica de Yucatán\Documentos\Carrera Academica\Data Engineer\7_Quadrimester\Internship\Validition-Pipeline-RAG-Law-and-Compliance-LGBTIQ-Rights\data\raw\Baja_California
PDFs found    : 1


## 3. Extract Text from Each PDF

For each PDF:
- Skip if the `.txt` output already exists (idempotent re-runs).
- Extract text page by page with `pdfplumber`.
- Save to `OUT_DIR/{stem}.txt` in UTF-8.
- Print page count and character count as a progress indicator.

In [8]:
extracted = 0
skipped   = 0
empty     = 0

for pdf_path in pdfs:
    out_path = OUT_DIR / f"{pdf_path.stem}.txt"

    if out_path.exists():
        print(f"[SKIP] {pdf_path.name} — already extracted")
        skipped += 1
        continue

    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = [page.extract_text() or "" for page in pdf.pages]
            text  = "\n\n".join(pages)
            n_pages = len(pdf.pages)
    except Exception as exc:
        print(f"[ERROR] {pdf_path.name}: {exc}")
        empty += 1
        continue

    if not text.strip():
        print(f"[EMPTY] {pdf_path.name} — pdfplumber returned no text ({n_pages} pages)")
        empty += 1
        continue

    out_path.write_text(text, encoding="utf-8")
    print(f"[OK]   {pdf_path.name} — {n_pages} pages, {len(text):,} chars → {out_path.name}")
    extracted += 1

[OK]   20250214_LEYCOMISIONDH.pdf — 38 pages, 88,977 chars → 20250214_LEYCOMISIONDH.txt


## 4. Summary

In [9]:
print("=" * 40)
print(f"Extracted successfully : {extracted}")
print(f"Skipped (already done) : {skipped}")
print(f"Empty / failed         : {empty}")
print(f"Total PDFs found       : {len(pdfs)}")

txt_files = list(OUT_DIR.glob("*.txt"))
print(f"\n.txt files in output dir: {len(txt_files)}")

Extracted successfully : 1
Skipped (already done) : 0
Empty / failed         : 0
Total PDFs found       : 1

.txt files in output dir: 8
